# M8 · Real expert offloading on the T4

Until now the cache was **simulated**. Here we really run OLMoE with:
- all 1024 experts (16 layers × 64) in **CPU memory**,
- only **K slots per layer on the GPU**, managed by the same `LayerCache` + LRU as the simulator,
- attention, router and embeddings (~1.3 GB) on the GPU as usual.

We measure GPU memory, decoding speed and energy, and check that the real
number of misses **equals** the simulator's for the same tokens.

**Kaggle:** GPU T4 x1, Internet On. ~1 hour. Needs the M3 output as input
(for the comparison table); the traces dataset is not needed.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

In [ ]:
import gc, os, time
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import config
from src.offload import offload_model, offload_stats
from src.baselines import decode_over_text
from src.energy import EnergyMeter
from src.simulator import simulate
from src.data import make_chunks

tok = AutoTokenizer.from_pretrained(config.MOE_MODEL)
text = make_chunks(tok, "wikitext")[0]          # 512 tokens of real English text
N_TOKENS = 256                                  # how many tokens to decode

## Run with K slots per layer
`run_offloaded(K)`:
1. loads OLMoE into **CPU** memory,
2. swaps every layer's experts for `OffloadedExperts` (GPU slot pool of size K),
3. decodes `N_TOKENS` tokens one at a time (like a chatbot),
4. measures speed, GPU memory and energy,
5. replays the logged expert choices in the simulator and compares the miss counts.

In [ ]:
def run_offloaded(slots):
    model = AutoModelForCausalLM.from_pretrained(config.MOE_MODEL, dtype=torch.float16,
                                                 low_cpu_mem_usage=True)       # on the CPU
    model = offload_model(model, slots)                                        # experts stay on CPU
    torch.cuda.reset_peak_memory_stats()
    print(f"K={slots}: GPU memory after loading: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    with EnergyMeter() as meter:
        decode_over_text(model, text[:N_TOKENS])
        torch.cuda.synchronize()

    stats = offload_stats(model)
    sim = simulate(stats["trace"], "LRU", slots)       # same expert choices, simulated
    row = {"slots": slots,
           "peak_GPU_GB": torch.cuda.max_memory_allocated() / 1e9,
           "decode_tok_per_s": N_TOKENS / meter.seconds,
           "decode_J_per_tok": meter.joules / N_TOKENS if meter.available else None,
           "hit_rate": stats["hit_rate"],
           "real_misses": stats["misses"], "simulated_misses": sim["misses"]}
    del model; gc.collect(); torch.cuda.empty_cache()  # free CPU and GPU memory for the next run
    return row

rows = [run_offloaded(16)]
pd.DataFrame(rows)

In [ ]:
rows.append(run_offloaded(32))
offload = pd.DataFrame(rows)
offload.to_csv(os.path.join(config.RESULTS_DIR, "offload.csv"), index=False)
offload

## Check: does the simulator tell the truth?
`real_misses` must equal `simulated_misses`. If it does, the M5–M7 simulation results
describe what a real offloading system would do with these policies.

In [ ]:
assert (offload.real_misses == offload.simulated_misses).all(), "simulator and real cache disagree!"
print("✅ real misses == simulated misses for every K")

## Compare with the dense models (from M3)
Add the **M3 notebook output** as an input so `baselines.csv` can be found.

In [ ]:
from src.files import find_file

base = pd.read_csv(find_file("baselines.csv"))
base["name"] = base["model"].str.split("/").str[-1]
table = base[["name", "peak_GPU_GB", "decode_tok_per_s", "decode_J_per_tok", "GFLOPs_per_token"]].copy()
moe_flops = base.loc[base.name.str.contains("OLMoE"), "GFLOPs_per_token"].item()
for _, r in offload.iterrows():
    table.loc[len(table)] = [f"OLMoE offloaded K={r.slots}", r.peak_GPU_GB, r.decode_tok_per_s,
                             r.decode_J_per_tok, moe_flops]
table.round(3)

## What to write in `docs/M8.md`
- GPU memory: offloaded OLMoE vs full OLMoE vs dense 7B.
- Speed: offloading is **slower** than keeping everything on the GPU (copies over PCIe).
  Say this honestly. The gain is memory (and compute vs dense 7B); the cache policy decides how much speed we lose.
- The simulator check passed (or not).